# GECS Classification — Exploratory Data Analysis
**DePaul x Morningstar Capstone | Spring 2026**

Run after cleaning. Set `TASK` and `INPUT_PATH` in Config before running.

## 0. Imports & Config

In [ ]:
import warnings
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', palette='muted')

# ── CONFIG ──────────────────────────────────────────
TASK        = 1        # 1 or 2
INPUT_PATH  = 'task1_gecs_classification_final_cleaned.csv'   # use cleaned file
# ────────────────────────────────────────────────────

LABEL_COL   = 'MstarGlobal' if TASK == 1 else 'SubIndustry'
TEXT_COLS   = ['LongProfile', 'SegmentName', 'SegmentDescription'] if TASK == 1 else ['SegmentName', 'SegmentDescription']

print(f'Task {TASK} | Label: {LABEL_COL}')

## 1. Load Data

In [ ]:
df = pd.read_csv(INPUT_PATH, encoding='utf-8')
df['AsOfDate'] = pd.to_datetime(df['AsOfDate'], errors='coerce')
print(f'Shape: {df.shape}')
df.head(3)

## 2. Class Distribution

In [ ]:
class_counts = df[LABEL_COL].value_counts()
n_classes = len(class_counts)

print(f'Total classes     : {n_classes}')
print(f'Total samples     : {len(df):,}')
print(f'Avg per class     : {len(df)/n_classes:.1f}')
print(f'Median per class  : {class_counts.median():.0f}')
print(f'Max (most common) : {class_counts.max():,} — {class_counts.idxmax()}')
print(f'Min (least common): {class_counts.min():,} — {class_counts.idxmin()}')
print(f'Imbalance ratio   : {class_counts.max()/class_counts.min():.1f}x')
print()
print(f'Classes with < 10 samples : {(class_counts < 10).sum()}')
print(f'Classes with < 30 samples : {(class_counts < 30).sum()}')
print(f'Classes with < 50 samples : {(class_counts < 50).sum()}')

In [ ]:
# Full class distribution — all classes sorted
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(n_classes), class_counts.values, color='#4a90d9', width=1.0)
ax.axhline(50, color='red', linestyle='--', linewidth=0.8, label='50 samples threshold')
ax.axhline(class_counts.median(), color='orange', linestyle='--', linewidth=0.8, label=f'Median ({class_counts.median():.0f})')
ax.set_title(f'Task {TASK} — All {n_classes} Classes by Sample Count (sorted desc)')
ax.set_xlabel('Class rank')
ax.set_ylabel('Sample count')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 and bottom 20
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

class_counts.head(20).sort_values().plot(kind='barh', ax=axes[0], color='#2ecc71')
axes[0].set_title(f'Top 20 Most Frequent Classes')
axes[0].set_xlabel('Sample Count')

class_counts.tail(20).sort_values().plot(kind='barh', ax=axes[1], color='#e74c3c')
axes[1].set_title(f'Bottom 20 Least Frequent Classes')
axes[1].set_xlabel('Sample Count')

plt.suptitle(f'Task {TASK} — Class Frequency Extremes', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Class size buckets
bins   = [0, 10, 30, 50, 100, 200, 500, float('inf')]
labels = ['<10', '10-30', '30-50', '50-100', '100-200', '200-500', '500+']
bucket_counts = pd.cut(class_counts, bins=bins, labels=labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
bucket_counts.plot(kind='bar', ax=ax, color='#9b59b6', edgecolor='white')
ax.set_title(f'Task {TASK} — Number of Classes by Sample Bucket')
ax.set_xlabel('Samples per class')
ax.set_ylabel('Number of classes')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(bucket_counts):
    ax.text(i, v + 0.3, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 3. Temporal Trends

In [ ]:
df['Year'] = df['AsOfDate'].dt.year

# Records per year
fig, ax = plt.subplots(figsize=(10, 4))
df['Year'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#2196a8')
ax.set_title(f'Task {TASK} — Records per Year')
ax.set_xlabel('Year')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 classes over time
top10 = class_counts.head(10).index.tolist()
trend_df = df[df[LABEL_COL].isin(top10)].groupby(['Year', LABEL_COL]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
trend_df.plot(ax=ax, marker='o', linewidth=1.5)
ax.set_title(f'Task {TASK} — Top 10 Classes: Sample Count by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Sample Count')
ax.legend(title=LABEL_COL, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Classes that appear in only 1 year vs multiple years
years_per_class = df.groupby(LABEL_COL)['Year'].nunique()
print('Years of data per class:')
print(years_per_class.value_counts().sort_index().to_string())

## 4. Text Length Analysis

In [ ]:
# Add character and word count columns
for col in TEXT_COLS:
    df[f'{col}_char_len'] = df[col].fillna('').str.len()
    df[f'{col}_word_len'] = df[col].fillna('').str.split().str.len()

# Summary stats
len_cols = [f'{c}_char_len' for c in TEXT_COLS]
print(df[len_cols].describe().round(1).to_string())

In [ ]:
# Character length distributions
fig, axes = plt.subplots(1, len(TEXT_COLS), figsize=(6*len(TEXT_COLS), 4))
if len(TEXT_COLS) == 1: axes = [axes]

colors = ['#4caf7d', '#e87c2a', '#4a90d9']
for ax, col, color in zip(axes, TEXT_COLS, colors):
    data = df[f'{col}_char_len']
    data[data > 0].plot(kind='hist', bins=40, ax=ax, color=color, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1, label=f'Median: {data.median():.0f}')
    ax.set_title(col)
    ax.set_xlabel('Character count')
    ax.legend()

plt.suptitle(f'Task {TASK} — Text Length Distributions (char count)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Average text length by class (top 30 classes only)
if TASK == 1:
    primary_col = 'LongProfile_char_len'
else:
    primary_col = 'SegmentDescription_char_len'

top30 = class_counts.head(30).index
avg_len = df[df[LABEL_COL].isin(top30)].groupby(LABEL_COL)[primary_col].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
avg_len.plot(kind='barh', ax=ax, color='#4a90d9')
ax.axvline(df[primary_col].mean(), color='red', linestyle='--', linewidth=1, label='Overall mean')
ax.set_title(f'Task {TASK} — Avg Text Length by Class (Top 30)')
ax.set_xlabel('Avg character count')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Top Keywords per Class (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use primary text column
primary_text = 'LongProfile' if TASK == 1 else 'SegmentDescription'

# Build corpus per class (concatenate all docs in each class)
top_n_classes = 10
top_classes = class_counts.head(top_n_classes).index.tolist()

class_corpus = {}
for cls in top_classes:
    texts = df[df[LABEL_COL] == cls][primary_text].dropna().tolist()
    class_corpus[cls] = ' '.join(texts)

# Fit TF-IDF
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)
corpus_list = list(class_corpus.values())
tfidf_matrix = tfidf.fit_transform(corpus_list)
feature_names = tfidf.get_feature_names_out()

print(f'Top 10 keywords per class ({primary_text}):') 
print('='*60)
for i, cls in enumerate(top_classes):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:10]
    keywords = [feature_names[j] for j in top_idx]
    print(f'\n{cls}: {keywords}')

In [ ]:
# Visualize top keywords for top 6 classes
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, cls in enumerate(top_classes[:6]):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:12]
    keywords = [feature_names[j] for j in top_idx]
    values   = [scores[j] for j in top_idx]

    axes[i].barh(keywords[::-1], values[::-1], color='#4a90d9')
    axes[i].set_title(f'Class: {cls}\n(n={class_counts[cls]:,})', fontsize=10)
    axes[i].set_xlabel('TF-IDF score')

plt.suptitle(f'Task {TASK} — Top Keywords per Class (TF-IDF)', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 6. Feature Relationships (Task 1 only)

In [ ]:
if TASK == 1:
    # Revenue share by class (top 20)
    top20 = class_counts.head(20).index
    rev_by_class = df[df[LABEL_COL].isin(top20)].groupby(LABEL_COL)['revenue_share'].mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    rev_by_class.plot(kind='bar', ax=ax, color='#9b59b6', edgecolor='white')
    ax.set_title('Avg Revenue Share by GECS Class (Top 20)')
    ax.set_xlabel('GECS Class')
    ax.set_ylabel('Avg Revenue Share')
    ax.axhline(df['revenue_share'].mean(), color='red', linestyle='--', linewidth=1, label=f'Overall mean: {df["revenue_share"].mean():.2f}')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Task 2 — no numeric revenue features to analyze.')

In [ ]:
if TASK == 1:
    # Single-segment vs multi-segment companies by class
    seg_mix = df.groupby(LABEL_COL)['is_largest_share_segment'].mean()
    top20_seg = seg_mix[class_counts.head(20).index].sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    top20_seg.plot(kind='bar', ax=ax, color='#e87c2a', edgecolor='white')
    ax.set_title('% Single-Segment Companies by GECS Class (Top 20)')
    ax.set_xlabel('GECS Class')
    ax.set_ylabel('Proportion is_largest_share_segment = True')
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 7. Class Overlap / Similarity Check

In [ ]:
# How many unique words does each class use?
# Helps identify classes that may be hard to distinguish

primary_text = 'LongProfile' if TASK == 1 else 'SegmentDescription'
top15 = class_counts.head(15).index.tolist()

vocab_sizes = {}
for cls in top15:
    texts = df[df[LABEL_COL] == cls][primary_text].dropna()
    words = ' '.join(texts).lower().split()
    vocab_sizes[cls] = len(set(words))

vocab_df = pd.Series(vocab_sizes).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
vocab_df.plot(kind='bar', ax=ax, color='#27ae60', edgecolor='white')
ax.set_title(f'Task {TASK} — Unique Vocabulary Size per Class (Top 15)')
ax.set_xlabel('Class')
ax.set_ylabel('Unique word count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. EDA Summary

In [ ]:
primary_text = 'LongProfile' if TASK == 1 else 'SegmentDescription'
primary_len  = f'{primary_text}_char_len'

print('=' * 55)
print(f'  EDA SUMMARY — Task {TASK}')
print('=' * 55)
print(f'  Total records         : {len(df):,}')
print(f'  Total classes         : {n_classes}')
print(f'  Avg samples/class     : {len(df)/n_classes:.1f}')
print(f'  Median samples/class  : {class_counts.median():.0f}')
print(f'  Imbalance ratio       : {class_counts.max()/class_counts.min():.1f}x')
print(f'  Classes < 50 samples  : {(class_counts < 50).sum()}')
print(f'  Primary text median   : {df[primary_len].median():.0f} chars')
print(f'  Date range            : {df["AsOfDate"].min().date()} to {df["AsOfDate"].max().date()}')
print('=' * 55)